Objectif : Reconstitué le dataSet avec les variables du paper Algerie

Les variable non contenus dans le dataSet Paper India:
Avg. Temperature ◦C

Distance River

Max Temperature ◦C

Min Temperature ◦C

Humidity %

Wind Speed

Annual Precip.

1. Imports

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
from shapely.geometry import Point
from tqdm import tqdm
from google.colab import drive

2. Montage Google Drive

In [ ]:
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


3. Chemins des fichiers

In [ ]:

dataSetPaperIndia  =  "/content/gdrive/MyDrive/Projet big data dossier organisé /dataTraiter/datasetPaperIndian.csv"
dataSetTemperature = "/content/gdrive/MyDrive/Projet big data dossier organisé /MehtodePaperAlg/soil_climat_fire_grid_Elev_merged.csv"
path_nc_wind       =  "/content/gdrive/MyDrive/Projet big data dossier organisé /MehtodePaperAlg/H_ERA5_ECMW_T639_WS-_0100m_Pecd_025d_S202301010000_E202312312300_INS_MAP_01h_COM_mbc_org_NA_NA---_NA---_NA---_PECD4.2_fv1.area-subset.42.15.-6.19.36.96.-9.5.nc"
path_nc_temp       =  "/content/gdrive/MyDrive/Projet big data dossier organisé /MehtodePaperAlg/2m_temperature_0_daily-mean.nc"
path_nc_dewpoint   =  "/content/gdrive/MyDrive/Projet big data dossier organisé /MehtodePaperAlg/2m_dewpoint_temperature_stream-oper_daily-mean.nc"
output_csv_final   ="/content/gdrive/MyDrive/Projet big data dossier organisé /dataTraiter/datasetPaperAlg.csv"
csv_path_distances = "/content/gdrive/MyDrive/Projet big data dossier organisé /dataTraiter/distances_grille_portugal.csv"


 4. Chargement et merge des datasets CSV

In [ ]:
dfIndia = pd.read_csv(dataSetPaperIndia)
df_distances = pd.read_csv(csv_path_distances)
dfIndia = dfIndia.drop(columns=["mean_LST_celsius", "mean_twi", "landcover"])

dfTemperaturePrecipitation = pd.read_csv(dataSetTemperature)

cols_to_merge = [
    "latitude", "longitude",
    "prec_automne", "prec_ete", "prec_hiver", "prec_printemps",
    "tmax_automne", "tmax_ete", "tmax_hiver", "tmax_printemps",
    "tmin_automne", "tmin_ete", "tmin_hiver", "tmin_printemps"
]

df_merged = pd.merge(
    dfIndia,
    dfTemperaturePrecipitation[cols_to_merge],
    on=["latitude", "longitude"],
    how="left"
)
df_merged=pd.merge(
    df_merged,
    df_distances[['latitude', 'longitude', 'dist_eau_m']],
    on=['latitude', 'longitude'],
    how='left'
)
print(f"Shape : {df_merged.shape}")
display(df_merged.head(3))

Shape : (89198, 23)


,latitude,longitude,in_fire,dist_fire_km,mean_slope,mean_aspect,ndvi_final,dist_residentiel_m,dist_route_m,Elevation,...,prec_printemps,tmax_automne,tmax_ete,tmax_hiver,tmax_printemps,tmin_automne,tmin_ete,tmin_hiver,tmin_printemps,dist_eau_m
0,36.987823,-7.963676,0,17.197693,0.571321,196.846283,0.686275,1846.893734,471.899104,0.0,...,50.800002,23.666667,28.666667,19.0,22.000000,15.666667,19.166667,10.0,12.333333,0.000000
1,36.994129,-8.952213,0,23.004464,2.499209,178.080078,0.866667,1447.688717,170.582454,0.0,...,61.241665,22.166667,24.833333,19.0,20.416667,16.166667,17.500000,11.5,12.916667,1815.680669
2,36.996850,-7.974890,0,16.054264,1.402498,179.751862,0.889725,457.222141,122.542919,0.0,...,50.800002,23.666667,28.666667,19.0,22.000000,15.666667,19.166667,10.0,12.333333,0.000000


5. Calcul Température Moyenne / Max / Min et Précipitation Annuelle

In [ ]:
saisons = ['automne', 'ete', 'hiver', 'printemps']

df_merged['avg_temperature'] = df_merged[[f'tmax_{s}' for s in saisons] + [f'tmin_{s}' for s in saisons]].mean(axis=1)
df_merged['max_temperature'] = df_merged[[f'tmax_{s}' for s in saisons]].max(axis=1)
df_merged['min_temperature'] = df_merged[[f'tmin_{s}' for s in saisons]].min(axis=1)
df_merged['annual_precip']   = df_merged[[f'prec_{s}' for s in saisons]].sum(axis=1)

cols_to_drop = [f'prec_{s}' for s in saisons] + [f'tmax_{s}' for s in saisons] + [f'tmin_{s}' for s in saisons]
df_merged = df_merged.drop(columns=cols_to_drop)

print(df_merged[['avg_temperature','max_temperature','min_temperature','annual_precip']].describe())

       avg_temperature  max_temperature  min_temperature  annual_precip
count     89198.000000     89198.000000     89198.000000   89198.000000
mean         16.547197        29.264416         6.740155     313.356951
std           1.721571         2.444923         1.858115     122.282625
min          10.302083        23.000000         0.500000     162.150004
25%          15.260417        27.166667         5.583333     205.899999
50%          17.177083        29.083333         7.000000     274.691667
75%          17.947917        31.583333         8.083333     419.891666
max          19.520833        34.416667        11.500000     648.800015


In [ ]:
print(df_merged[['avg_temperature','max_temperature','min_temperature','annual_precip']].head())

   avg_temperature  max_temperature  min_temperature  annual_precip
0          18.8125        28.666667             10.0     162.500003
1          18.0625        24.833333             11.5     168.908329
2          18.8125        28.666667             10.0     162.500003
3          18.0625        24.833333             11.5     168.908329
4          18.0625        24.833333             11.5     168.908329


6. Calcul Humidité Relative (ERA5)